# Phase 06B.04 — Novelty Statistical Analysis & Winner Lock

Validate preregistered full prediction artifacts, run paired group-cluster inference, apply Holm correction, and lock the winner.

**Immutable gates:** `L32-F1`; 298 frozen validation samples; train-side checkpoint selection only; no public-test access. Missing human/input artifacts produce an explicit status and stop—no synthetic labels or provenance.

## 1. Protocol and experiment registry gate

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
def write_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path
from phase06a_common import compute_classification_metrics,exact_mcnemar,holm_adjust,paired_group_cluster_bootstrap,sha256_file,sha256_json,validate_prediction_artifact
from phase06b_common import LOCKED_BASELINE_WINNER,load_json,validate_locked_baseline
P=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; R=ROOT/"outputs/phase06b/experiments/novelty_experiment_registry.json"
OUT=ROOT/"outputs/phase06b/final_analysis"; IDS=ROOT/"data/splits/phase01/validation_sample_ids.json"; OUT.mkdir(parents=True,exist_ok=True)
baseline_manifest=validate_locked_baseline(ROOT)
if not P.is_file(): raise RuntimeError("Novelty protocol is not locked")
protocol=load_json(P)

In [ ]:
if not R.is_file():
 write_json(OUT/"novelty_experiment_registry.template.json",{"status":"complete","selected_track":protocol["selected_track"],
 "protocol_sha256":sha256_json(protocol),"control":{"name":"L32-F1","predictions_path":baseline_manifest["winner_predictions"]["path"],
 "sha256":baseline_manifest["winner_predictions"]["sha256"]},"candidates":[{"name":"","predictions_path":"","sha256":""}]})
 write_json(OUT/"PHASE06B_04_STATUS.json",{"status":"awaiting_experiments"})
 raise RuntimeError("Provide preregistered full novelty predictions")
registry=load_json(R)
if registry.get("status")!="complete" or registry.get("selected_track")!=protocol.get("selected_track") or registry.get("protocol_sha256")!=sha256_json(protocol):
 raise ValueError("Registry does not match locked protocol")
if registry.get("control",{}).get("name")!=LOCKED_BASELINE_WINNER or not registry.get("candidates"): raise ValueError("Registry needs L32-F1 and candidates")
if any("oracle" in str(x.get("name","")).casefold() for x in registry["candidates"]): raise ValueError("Oracle cannot enter winner selection")

## 2. Integrity and paired statistics

In [ ]:
ids=json.loads(IDS.read_text())
def load_arm(item):
 path=Path(item["predictions_path"]); path=path if path.is_absolute() else ROOT/path
 if not path.is_file() or sha256_file(path)!=item["sha256"]: raise ValueError(f"Artifact mismatch: {item['name']}")
 frame=pd.read_csv(path); validate_prediction_artifact(frame,ids,run_scope="full"); return frame
control=load_arm(registry["control"]); candidates={x["name"]:load_arm(x) for x in registry["candidates"]}
leader=[]; summaries={}; raw={}
for name,frame in [(LOCKED_BASELINE_WINNER,control),*candidates.items()]:
 m=compute_classification_metrics(frame); leader.append({"experiment":name,**{k:m[k] for k in ["accuracy","macro_f1","parse_rate"]}})
for i,(name,frame) in enumerate(candidates.items()):
 dist,summary=paired_group_cluster_bootstrap(control,frame,resamples=10_000,seed=42+i); dist.to_csv(OUT/f"bootstrap_{name}_vs_L32-F1.csv",index=False)
 left=control.sort_values("sample_id").correct.astype(bool); right=frame.sort_values("sample_id").correct.astype(bool)
 test=exact_mcnemar(left,right); summaries[name]=summary; raw[name]=test["exact_two_sided_p"]
adj=holm_adjust(raw); rows=[]
for name in candidates:
 d=summaries[name]["accuracy_delta"]; rows.append({"experiment":name,"accuracy_delta_mean":d["mean"],"ci95_low":d["ci95_low"],"ci95_high":d["ci95_high"],"mcnemar_raw_p":raw[name],"mcnemar_holm_p":adj[name],"qualifies":d["ci95_low"]>0 and adj[name]<.05})
leader=pd.DataFrame(leader).sort_values(["accuracy","macro_f1"],ascending=False); stats=pd.DataFrame(rows).sort_values("accuracy_delta_mean",ascending=False)
leader.to_csv(OUT/"novelty_leaderboard.csv",index=False); stats.to_csv(OUT/"novelty_paired_statistics.csv",index=False)
stats

## 3. Deterministic winner lock
If no novelty arm clears both preregistered gates, L32-F1 remains winner.

In [ ]:
qualified=stats[stats.qualifies]
winner=LOCKED_BASELINE_WINNER if qualified.empty else qualified.iloc[0].experiment
reason="No novelty arm cleared both gates." if qualified.empty else "Highest delta among arms clearing paired CI and Holm gates."
manifest={"status":"complete","winner":winner,"baseline_control":LOCKED_BASELINE_WINNER,"selected_track":protocol["selected_track"],
 "decision":reason,"protocol_sha256":sha256_json(protocol),"registry_sha256":sha256_file(R),"validation_rows":len(ids),
 "bootstrap_resamples":10_000,"holm_alpha":.05,"public_test_accessed":False}
write_json(OUT/"novelty_winner_manifest.json",manifest); write_json(OUT/"PHASE06B_04_STATUS.json",{"status":"complete","winner":winner})
manifest